In [2]:
from fom import create_fom
import numpy as np

fom = create_fom(nt=300, h=50, time=9)

print(fom)

InstationaryModel
    class: InstationaryModel
    non-linear
    T: 9
    solution_space:  NGSolveVectorSpace(<ngsolve.comp.ProductSpace object at 0x000001AD78D1EE10>)
    dim_input:       0
    dim_output:      0


In [3]:
# access parameter study

from pymor.core.pickle import load

with open('brusselator_parameters.pkl', 'rb') as f:
    parameterstudy = load(f)

print(f"Mus we got: {list(parameterstudy.keys())}")
print("Currently safed in parameterstudy:")
for m, data in parameterstudy.items():
    print(f"{m} -> T = {data['period']:.4f}")

Mus we got: [2.75, 3.25, 3.75, 4.25]
Currently safed in parameterstudy:
2.75 -> T = 6.7286
3.25 -> T = 7.3714
3.75 -> T = 8.3143
4.25 -> T = 9.4714


In [4]:
beta = fom.parameters.parse({'beta': 3.0})
U_trajectory = fom.solve(beta)

Accordion(children=(HTML(value='', layout=Layout(height='16em', width='100%')),), titles=('Log Output',))

In [ ]:
# compute the customized snapshots

import gc
import numpy as np
customized_snapshots = fom.solution_space.empty()
M=65

for mu, data in parameterstudy.items():
    T_per = data['period']
    u0_array = data['initial_data']
    
    print(f"Compute = {mu}")
    
    fom_snapshot = create_fom(nt=300, h=50, time=T_per, initial_array=u0_array) # customized fom
    
    U_snapshot = fom_snapshot.solve(mu)
    
    N = len(U_snapshot)
    indices = np.linspace(0, N - 1, M, dtype=int)  # we want M snapshots per parameter
    numpy_daten = U_snapshot[indices].to_numpy()
    kompatible_snapshots = fom.solution_space.from_numpy(numpy_daten)
    customized_snapshots.append(kompatible_snapshots)

    del U_snapshot
    del fom_snapshot
    gc.collect()
    
len(customized_snapshots) # nt*number_sample_mus, 260



In [8]:
# safe the customized_snapshot to access in second notebook

from pymor.core.pickle import dump

with open('customized_brusselator_snapshots.pkl', 'wb') as f:
    dump(customized_snapshots, f)

In [4]:
# access customized snapshots

from pymor.core.pickle import load

with open('customized_brusselator_snapshots.pkl', 'rb') as f:
    alte_snapshots1 = load(f)
customized_snapshots = fom.solution_space.from_numpy(alte_snapshots1.to_numpy())
 
# access parameter study
with open('brusselator_parameters.pkl', 'rb') as f:
    parameterstudy = load(f)

   
len(customized_snapshots)

260

In [12]:
# create curly U snapshots, customized version, DtDa

U_curly_DtDa = fom.solution_space.empty() 

sample_mus = [2.75, 3.25, 3.75, 4.25]
L = len(sample_mus) - 1
M = 64
N = (L + 1) * (M + 1)
d_alpha = sample_mus[1] - sample_mus[0]

all_solutions = []
all_dts = []    # since we have different final times, we have different dt aswell

for i, mu in enumerate(sample_mus):

    start = i * (M + 1)
    end = (i + 1) * (M + 1)
    
    sol_mu = customized_snapshots[start:end].copy()   # we take the solutions we already computed in std_snapshots
   
    raw_data = sol_mu.to_numpy()            # this is just to make the pickle data compatible
    sol_mu = fom.solution_space.from_numpy(raw_data)
    
    all_solutions.append(sol_mu)
    
    T_mu = parameterstudy[mu]['period']  
    all_dts.append(T_mu / M)

for l in range(L + 1):
    vector = all_solutions[l][0].copy()
    vector *= np.sqrt(N)
    U_curly_DtDa.append(vector)

for j in range(1, M + 1):
    dt_0 = all_dts[0]
    diff_t = (all_solutions[0][j] - all_solutions[0][j-1]) * (1.0 / dt_0)
    diff_t *= np.sqrt(L + 1)
    U_curly_DtDa.append(diff_t)

for l in range(1, L + 1):
    dt_l = all_dts[l]
    for j in range(1, M + 1):
        u_alpha_diff_j = (all_solutions[l][j] - all_solutions[l-1][j]) * (1.0 / d_alpha)
        u_alpha_diff_j_minus_1 = (all_solutions[l][j-1] - all_solutions[l-1][j-1]) #* (1.0 / d_alpha)
        
        mixed_diff = (u_alpha_diff_j - u_alpha_diff_j_minus_1)# * (1.0 / dt_l)
        U_curly_DtDa.append(mixed_diff)

len(U_curly_DtDa)

260

In [13]:
# create curly U snapshots, customized version, DaDt

U_curly_DaDt = fom.solution_space.empty() 

sample_mus = [2.75, 3.25, 3.75, 4.25]
L = len(sample_mus) - 1
M = 64
N = (L + 1) * (M + 1)
d_alpha = sample_mus[1] - sample_mus[0]

all_solutions = []
all_dts = []    # since we have different final times, we have different dt aswell

for i, mu in enumerate(sample_mus):

    start = i * (M + 1)
    end = (i + 1) * (M + 1)
    
    sol_mu = customized_snapshots[start:end].copy()   # we take the solutions we already computed in std_snapshots
   
    raw_data = sol_mu.to_numpy()                   # this is just to make the pickle data compatible
    sol_mu = fom.solution_space.from_numpy(raw_data)
    
    all_solutions.append(sol_mu)
    
    T_mu = parameterstudy[mu]['period']  
    all_dts.append(T_mu / M)

for l in range(L + 1):
    vector = all_solutions[l][0].copy()
    vector *= np.sqrt(N)
    U_curly_DaDt.append(vector)

for j in range(1, M + 1):
    dt_0 = all_dts[0]
    diff_t = (all_solutions[0][j] - all_solutions[0][j-1]) * (1.0 / dt_0)
    diff_t *= np.sqrt(L + 1)
    U_curly_DaDt.append(diff_t)

for l in range(1, L + 1):
    dt_l = all_dts[l]
    dt_l_minus_1 = all_dts[l-1]
    
    for j in range(1, M + 1):
        u_t_diff_l = (all_solutions[l][j] - all_solutions[l][j-1]) * (1.0 / dt_l)
        u_t_diff_l_minus_1 = (all_solutions[l-1][j] - all_solutions[l-1][j-1])# * (1.0 / dt_l_minus_1)

        mixed_diff = (u_t_diff_l - u_t_diff_l_minus_1) #* (1.0 / d_alpha)
        U_curly_DaDt.append(mixed_diff)

len(U_curly_DaDt)

260

In [14]:
# safe the U_curly snapshots to access in second notebook

from pymor.core.pickle import dump

with open('curly_U_snapshots.pkl', 'wb') as f:
    dump(U_curly_DtDa, f)

with open('curly_U_DaDt_snapshots.pkl', 'wb') as f:
    dump(U_curly_DaDt, f)